### CIFAR-10分类
使⽤课上的两种不同的深度学习框架（Tensorflow/PyTorch/MindSpore三选⼆）分别搭建分类模型，在CIFAR-10数据集上完成分类任务。在实验报告中，需要⾄少给出以下内容：  
1）模型解释；  
2）训练集和测试集上的loss曲线；  
3）测试集上的accuarcy曲线；  
4）⽐较不同框架分类任务实现的性能指标、时间效率和开发难度。其中，性能指标使⽤测试集上的平均loss和分类准确率accuracy来衡量，时间效率使⽤Linux系统的time函数来计量或在python程序内部⾃⾏计量。

In [8]:
# 导入必要的库
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
import time
import wandb

wandb.init(
    project="homework",
    name="ex6-pytorch",

    config={
    "learning_rate": 0.001,
    "architecture": "CNN",
    "dataset": "CIFAR-10",
    "epochs": 10,
    }
)

# 超参数设置
num_epochs = 10
batch_size = 64
learning_rate = 0.001

# CIFAR-10 数据集
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=False)
test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform, download=False)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# 构建卷积神经网络
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = ConvNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 记录训练过程
train_losses, test_losses, accuracies = [], [], []

# 训练与测试过程
start_time = time.time()
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    model.eval()
    test_loss, correct = 0.0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()

    test_loss /= len(test_loader)
    test_losses.append(test_loss)
    accuracy = 100 * correct / len(test_dataset)
    wandb.log({"Train Loss": train_loss, "Test Loss": test_loss, "Test Accuracy": accuracy})
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}, Accuracy: {accuracy:.2f}%")

end_time = time.time()
print(f"Training Time: {end_time - start_time:.2f} seconds")



wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: violetever (violetever-tsinghua). Use `wandb login --relogin` to force relogin


Epoch [1/10], Train Loss: 1.3323, Test Loss: 1.0652, Accuracy: 62.35%
Epoch [2/10], Train Loss: 0.9455, Test Loss: 0.9079, Accuracy: 68.18%
Epoch [3/10], Train Loss: 0.7877, Test Loss: 0.8438, Accuracy: 70.57%
Epoch [4/10], Train Loss: 0.6747, Test Loss: 0.8355, Accuracy: 71.17%
Epoch [5/10], Train Loss: 0.5644, Test Loss: 0.8409, Accuracy: 71.79%
Epoch [6/10], Train Loss: 0.4714, Test Loss: 0.8768, Accuracy: 72.38%
Epoch [7/10], Train Loss: 0.3821, Test Loss: 0.9353, Accuracy: 72.58%
Epoch [8/10], Train Loss: 0.3028, Test Loss: 0.9668, Accuracy: 72.71%
Epoch [9/10], Train Loss: 0.2307, Test Loss: 1.1014, Accuracy: 71.83%
Epoch [10/10], Train Loss: 0.1796, Test Loss: 1.1959, Accuracy: 71.85%
Training Time: 335.50 seconds


In [2]:
import tensorflow as tf
import numpy as np
import time
import wandb
from tensorflow.keras import layers, models
import os
import pickle

# 初始化wandb
wandb.init(
    project="homework",
    name="ex6-tensorflow",

    config={
    "learning_rate": 0.001,
    "architecture": "CNN",
    "dataset": "CIFAR-10",
    "epochs": 10,
    }
)

# 定义从本地加载CIFAR-10数据集的函数
def load_cifar10_data(data_dir):
    def unpickle(file):
        with open(file, 'rb') as fo:
            dict = pickle.load(fo, encoding='bytes')
        return dict

    # 读取训练数据
    x_train, y_train = [], []
    for i in range(1, 6):
        batch = unpickle(os.path.join(data_dir, f"data_batch_{i}"))
        x_train.append(batch[b'data'])
        y_train += batch[b'labels']
    x_train = np.concatenate(x_train).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype("float32") / 255.0
    y_train = np.array(y_train)

    # 读取测试数据
    test_batch = unpickle(os.path.join(data_dir, "test_batch"))
    x_test = test_batch[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype("float32") / 255.0
    y_test = np.array(test_batch[b'labels'])

    return (x_train, y_train), (x_test, y_test)

# 指定数据路径并加载数据
data_dir = './data/cifar-10-batches-py'
(x_train, y_train), (x_test, y_test) = load_cifar10_data(data_dir)

# 定义分类模型
def build_model():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

model = build_model()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 使用回调记录loss和accuracy
class WandbCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        wandb.log({
            'train_loss': logs['loss'],
            'train_accuracy': logs['accuracy'],
            'val_loss': logs['val_loss'],
            'val_accuracy': logs['val_accuracy']
        })

# 计时并训练模型
start_time = time.time()
history = model.fit(x_train, y_train, epochs=10, 
                    validation_data=(x_test, y_test),
                    callbacks=[WandbCallback()])
end_time = time.time()

# 记录总的训练时间
training_time = end_time - start_time
print(f"Total training time: {training_time:.2f} seconds")
wandb.log({'total_training_time': training_time})

# 测试集性能评估
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc}")

wandb.finish()


  4079616/170498071 [..............................] - ETA: 13:33

KeyboardInterrupt: 